### Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from groq import Groq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv('D:/Semantic-Ecommerce-Recommender/.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if GROQ_API_KEY:
    print("API key loaded successfully!")
else:
    print("ERROR: API key not found. Check your .env file.")

# Load data
df = pd.read_csv('D:/Semantic-Ecommerce-Recommender/data/processed/amazon_featured.csv')
print(f"Dataset shape: {df.shape}")

API key loaded successfully!
Dataset shape: (8000, 20)


### Initialize Groq Client and Test

In [3]:
client = Groq(api_key=GROQ_API_KEY)

# Quick test with updated model
test_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50
)

print("LLM Test Response:")
print(test_response.choices[0].message.content)

LLM Test Response:
Hello, how are you today?


### Rebuild TF-IDF Recommender

In [4]:
# Rebuild TF-IDF from Week 3
df['combined_text'] = df['title'] + ' ' + df['category_name']

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df['combined_text'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

def recommend_tfidf(query, top_n=5):
    query_vec = tfidf.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]
    results = df.iloc[top_indices][['title', 'category_name', 'stars', 'price']].copy()
    results['similarity_score'] = similarities[top_indices].round(4)
    return results

print("TF-IDF recommender ready!")

TF-IDF matrix shape: (8000, 5000)
TF-IDF recommender ready!


### Prompt Engineering: Product Summary

In [6]:
def generate_product_summary(query, recommendations):
    """LLM generates a plain-English summary of recommendations."""
    
    products_text = ""
    for i, (_, row) in enumerate(recommendations.iterrows(), 1):
        products_text += f"{i}. {row['title']} | Category: {row['category_name']} | Stars: {row['stars']} | Price: ${row['price']}\n"
    
    system_prompt = """You are a helpful e-commerce shopping assistant. 
    When given a user's search query and a list of recommended products, 
    you provide a brief, friendly 2-3 sentence summary explaining why 
    these products match what the user is looking for. Be specific and helpful."""
    
    user_prompt = f"""
    User searched for: "{query}"
    
    Top recommended products:
    {products_text}
    
    Please provide a brief 2-3 sentence summary explaining these recommendations.
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=200
    )
    
    return response.choices[0].message.content

# Test it
query = "wireless headphones noise cancelling"
recs = recommend_tfidf(query, top_n=5)
print("Recommendations:")
print(recs[['title', 'category_name', 'stars', 'price']])
print("\nLLM Summary:")
print(generate_product_summary(query, recs))

Recommendations:
                                                  title         category_name  \
1880  Gsoemon Active Noise Cancelling Wireless Earbu...  Headphones & Earbuds   
7525  Active Noise Cancelling Headphones E600Pro, 80...  Headphones & Earbuds   
102   Wireless Ear Clip Bone Conduction Headphones B...  Headphones & Earbuds   
7948  Wireless Earbuds,Bluetooth 5.3 Headphones Buil...  Headphones & Earbuds   
5766  MGGZXR Wireless Neckband Earbuds with TF Card ...  Headphones & Earbuds   

      stars  price  
1880    4.5  29.62  
7525    4.0  99.90  
102     2.9  14.99  
7948    4.6  13.99  
5766    3.9  18.98  

LLM Summary:
Based on your search for wireless headphones with noise-cancelling capabilities, we've curated a list of top-notch options for you. 

You may like the **Gsoemon Active Noise Cancelling Wireless Earbuds** which feature high-quality noise cancellation and long playtime, making it perfect for commuting, traveling, or exercising ($29.62). The **Active Noise 

### Prompt Engineering: Investment/Value Analysis

In [7]:
def generate_value_analysis(query, recommendations):
    """LLM identifies the best value product from recommendations."""
    
    products_text = ""
    for i, (_, row) in enumerate(recommendations.iterrows(), 1):
        products_text += f"{i}. {row['title']} | Stars: {row['stars']} | Price: ${row['price']} | Category: {row['category_name']}\n"
    
    system_prompt = """You are a smart shopping advisor who helps users 
    find the best value for money. Analyze products and give clear, 
    actionable advice in 2-3 sentences. Always mention which product 
    offers the best value and why."""
    
    user_prompt = f"""
    User is looking for: "{query}"
    
    Available products:
    {products_text}
    
    Which product offers the best value for money? Explain why in 2-3 sentences.
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=200
    )
    
    return response.choices[0].message.content

# Test
print("Value Analysis:")
print(generate_value_analysis(query, recs))

Value Analysis:
The best value for money in wireless noise-cancelling headphones is the Gsoemon Active Noise Cancelling Wireless Earbuds. This product offers excellent noise cancellation with a 35dB rating, 24H playtime, and multiple features at an affordable price of $29.62, which is significantly cheaper than the other options. Its star rating of 4.5 also suggests a high level of customer satisfaction, making it a well-rounded choice for its price.


### Prompt Engineering: Shopping Tip Generator

In [9]:
def generate_shopping_tip(query, recommendations):
    """LLM generates a personalized shopping tip based on the query."""
    
    categories = recommendations['category_name'].unique().tolist()
    avg_price = recommendations['price'].mean()
    avg_stars = recommendations['stars'].mean()
    
    system_prompt = """You are a friendly shopping assistant that gives 
    practical, specific shopping tips. Keep tips to 2 sentences maximum. 
    Be helpful and specific to the product category."""
    
    user_prompt = f"""
    A user searched for: "{query}"
    Results are from categories: {categories}
    Average price: ${avg_price:.2f}
    Average rating: {avg_stars:.1f} stars
    
    Give them one practical shopping tip for buying this type of product.
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=150
    )
    
    return response.choices[0].message.content

# Test with multiple queries
test_queries = [
    "laptop for video editing",
    "yoga mat non slip",
    "coffee maker automatic"
]

for q in test_queries:
    recs = recommend_tfidf(q, top_n=5)
    print(f"\nQuery: '{q}'")
    print(f"Tip: {generate_shopping_tip(q, recs)}")



Query: 'laptop for video editing'
Tip: When shopping for a laptop for video editing, consider the processor's core count and speed, looking for at least a quad-core processor with a speed of 2.5 GHz or higher, as this will help you efficiently handle resource-intensive video editing tasks.

Query: 'yoga mat non slip'
Tip: When buying a non-slip yoga mat, consider the surface type you'll be practicing on. A mat with grippy textures that work well on smooth floors will be sufficient, whereas a mat with extra aggressive grippers will be ideal for use on slippery or wooden surfaces.

Query: 'coffee maker automatic'
Tip: When shopping for an automatic coffee maker, consider the number of cups it can brew at once and whether it fits your daily coffee needs in one go. Look for models that offer programmable timers and scheduling features to save you time in the mornings.


### Combined Pipeline Function

In [14]:
def full_recommendation_pipeline(query, top_n=5):
    """Complete pipeline: search + LLM insights."""
    
    print(f"Searching for: '{query}'")
    
    # Get recommendations
    recs = recommend_tfidf(query, top_n=top_n)
    
    print("\n Top Recommendations:")
    print(recs[['title', 'category_name', 'stars', 'price']].to_string(index=False))
    
    print("\n AI Summary:")
    print(generate_product_summary(query, recs))
    
    print("\n Value Analysis:")
    print(generate_value_analysis(query, recs))
    
    print("\n Shopping Tip:")
    print(generate_shopping_tip(query, recs))
    
    return recs

# Test the full pipeline
results = full_recommendation_pipeline("bluetooth speaker waterproof")

Searching for: 'bluetooth speaker waterproof'

 Top Recommendations:
                                                                                                                                                                                                title                category_name  stars  price
Bluetooth Receiver/Hands-Free Car Kit, Esuper Portable 3.5mm Bluetooth Aux Adapter Wireless Music Streaming for Home, Car Audio System, Headphone, Speaker(Bluetooth 4.2,A2DP,40feet Bluetooth Range)          Vehicle Electronics    4.2  10.19
                                                                                                                                                  16AWG Gauge Speaker Cable Speaker Wire Black (50FT) Televisions & Video Products    4.6  12.99
                   Cambridge Soundworks OontZ Angle 3 Ultra SUP Special Edition Waterproof Paddleboard Bluetooth Speaker, 14 Watts, Hi-Quality Sound & Bass, 100 Ft Wireless Range Bluetooth Speakers       Port

### Log API Usage

In [12]:
# Document API usage for submission
api_log = {
    "model": "llama-3.1-8b-instant",
    "provider": "Groq",
    "prompts_designed": [
        "Product Summary Generator",
        "Value Analysis Advisor", 
        "Shopping Tip Generator"
    ],
    "avg_tokens_per_call": "~200",
    "total_test_calls": 10,
    "week": 5
}

import json
with open('D:/Semantic-Ecommerce-Recommender/docs/api_usage_log.json', 'w') as f:
    json.dump(api_log, f, indent=2)

print("API usage log saved!")
print(json.dumps(api_log, indent=2))

API usage log saved!
{
  "model": "llama-3.1-8b-instant",
  "provider": "Groq",
  "prompts_designed": [
    "Product Summary Generator",
    "Value Analysis Advisor",
    "Shopping Tip Generator"
  ],
  "avg_tokens_per_call": "~200",
  "total_test_calls": 10,
  "week": 5
}


### Saving prompt engineering doc

In [13]:
prompt_doc = """
# Prompt Engineering Documentation
## Week 5 - Semantic E-Commerce Recommender

## Model Used
- Provider: Groq
- Model: llama-3.1-8b-instant
- Reason: Free tier, fast inference, strong instruction following

## Prompts Designed

### 1. Product Summary Generator
- Purpose: Explain why recommendations match the user query
- Approach: System prompt sets assistant as shopping helper,
  user prompt passes query + product list
- Output: 2-3 sentence natural language summary

### 2. Value Analysis Advisor  
- Purpose: Identify best value product from recommendations
- Approach: System prompt sets assistant as value advisor,
  user prompt passes products with price/rating data
- Output: 2-3 sentence value recommendation

### 3. Shopping Tip Generator
- Purpose: Give practical category-specific shopping advice
- Approach: System prompt sets assistant as practical advisor,
  user prompt passes category and price statistics
- Output: 1-2 sentence actionable tip

## Key Prompt Engineering Principles Applied
- Clear role definition in system prompt
- Specific output format requested (2-3 sentences)
- Relevant context passed (prices, ratings, categories)
- Temperature kept at default for consistent outputs
"""

with open('D:/Semantic-Ecommerce-Recommender/docs/prompt_engineering_doc.md', 'w') as f:
    f.write(prompt_doc)

print("Prompt engineering doc saved!")

Prompt engineering doc saved!
